## Building a BPE tokenizer

**Author:** Diego Besada

Created using [Pierre Nugues' template](https://github.com/pnugues/edan20/blob/master/labs_2026/3-BPE.ipynb)

BPE is a **subword tokenization** technique: starting from a basic set of
symbols, it repeatedly merges the most frequent pair of adjacent symbols into a
new one. After `k` merges, the vocabulary holds the original symbols plus the
most common subwords, and tokenizing means applying those merges in order.

In this notebook we work at the **character** level (not bytes), so the code
stays simple, and we first use the **`sentencepiece`** library as a reference.
Then we build our own tokenizer and compare it with the library's output.

1. Load the corpus.
2. Train a reference BPE with `sentencepiece`.
3. Implement BPE from scratch.
4. Tokenize and compare.

In [1]:
import os, requests, regex as re
from collections import defaultdict
from zipfile import ZipFile
import sentencepiece as spm

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
SELMA_URL = 'https://github.com/pnugues/ilppp/raw/master/programs/corpus/Selma.zip'
HERRGARD = os.path.join('Selma', 'herrgard.txt')

if not os.path.exists(HERRGARD):
    req = requests.get(SELMA_URL, stream=True)
    if req.status_code != 200:
        req.close()
        raise Exception(req.status_code)

    with open ('Selma.zip', 'wb') as fd:
        for chunk in req.iter_content(chunk_size=65536):
            fd.write(chunk)

    req.close()

    selma_zipfile = ZipFile('Selma.zip')
    selma_zipfile.extract(HERRGARD)
    selma_zipfile.close()
    os.remove('Selma.zip')

with open(HERRGARD, encoding='utf8') as f:
    corpus_raw: str = f.read().strip()

#### BPE (from `sentencepiece` library)

We first train a reference BPE model on `herrgard.txt` with `sentencepiece` and encode a sample string with it; its output is the target our own implementation must reproduce.

In [3]:
_ = spm.SentencePieceTrainer.train(
    input='Selma/herrgard.txt', 
    model_prefix='m', 
    vocab_size=116,
    model_type='BPE', 
    user_defined_symbols='0,1,2,3,4,5,6,7,8,9',
    minloglevel=3
)

sp = spm.SentencePieceProcessor()
_ = sp.load('m.model')

sp.encode('Selma Lagerlöf')
sp.encode('Selma Lagerlöf', out_type=str)

[63, 96, 64, 71, 76, 65, 63, 110, 65, 75, 37, 71, 84, 78]

['▁', 'S', 'e', 'l', 'm', 'a', '▁', 'L', 'a', 'g', 'er', 'l', 'ö', 'f']

#### BPE design

Our implementation follows the algorithm step by step, starting from single characters and merging the most frequent pair of adjacent symbols `k` times.

1) Replace all whitespace characters with `u2581` (`▁`) character. This is the character used by `sentencepiece` to represent whitespace.

In [4]:
corpus = '\u2581' + re.sub(r'\s+', '\u2581', corpus_raw)
corpus[:15]

'▁Selma▁Lagerlöf'

2. Split the corpus into a list of characters

In [5]:
corpus_l: list[str] = list(corpus)
corpus_l[:15]


['▁', 'S', 'e', 'l', 'm', 'a', '▁', 'L', 'a', 'g', 'e', 'r', 'l', 'ö', 'f']

3. Get the initial vocabulary

In [6]:
def get_initial_vocab(corpus: list[str]) -> set[str]:
    return set(corpus)

4. Count the frequency of all pairs of consecutive symbols in the corpus

In [7]:
def pair_count(corpus: list[str]) -> dict[tuple[str, str], int]:
    pairs = defaultdict(int)
    for i in range(len(corpus) - 1):
        left, right = corpus[i], corpus[i + 1]
        if '\u2581' in right:
            continue
        pairs[(left, right)] += 1
    return dict(pairs)

5. Expand the vocabulary by iteratively merging the most frequent pairs of tokens

In [8]:
def merge_bigrams(corpus: list[str], pair: tuple[str, str]) -> list[str]:
    new_corpus = []
    new_token = ''.join(pair)
    i = 0
    n = len(corpus)
    while i < n:
        if i < n - 1 and corpus[i] == pair[0] and corpus[i + 1] == pair[1]:
            new_corpus.append(new_token)
            i += 2
        else:
            new_corpus.append(corpus[i])
            i += 1
    return new_corpus

merge_bigrams(['T', 'h', 'i', 's', ' ', 'i', 's', ' ', 'a', '\u2581', 't', 'e', 's', 't'], ('i', 's'))

['T', 'h', 'is', ' ', 'is', ' ', 'a', '▁', 't', 'e', 's', 't']

In [9]:
def BPE(corpus: list[str], k: int) -> tuple[set[str], list[tuple[str, str]]]:
    '''Performs k iterations of the BPE algorithm on the given corpus.'''
    vocabulary: set[str] = get_initial_vocab(corpus)
    merge_ops: list[tuple[str, str]] = []

    for _ in range(k):
        pairs = pair_count(corpus)
        if not pairs:
            break
        most_freq_pair: tuple[str, str] = max(pairs, key=lambda pair: pairs[pair])
        new_symbol: str = ''.join(most_freq_pair)
        vocabulary.add(new_symbol)
        merge_ops.append(most_freq_pair)
        corpus = merge_bigrams(corpus, most_freq_pair)
        
    return vocabulary, merge_ops

vocabulary, merge_ops = BPE(corpus_l, 50)
print(vocabulary)
print(merge_ops)

{'é', 'U', '▁m', 'ig', '’', 'ör', '»', 'v', '▁en', 'G', 'r', 'k', 'Ö', '▁p', 'a', 'R', '▁b', '.', 'de', 'D', 'M', 'ch', 'är', '▁e', 'd', 'c', '!', '▁d', 'om', '?', 'tt', ';', 'll', '▁H', 'st', 'x', 'et', '▁det', 'C', '▁och', '▁k', 'y', '▁', 'na', '9', 'i', 'ä', 'p', '▁hon', 'b', 'f', ',', 'n', 'o', '-', 'on', 'än', 'O', '▁att', '▁de', 'å', 'X', 'g', 'Ä', 'an', 'h', '▁h', '▁o', '▁t', '▁n', '▁han', 'ö', 'fv', 'l', 'T', 'e', '1', 'Å', '_', '▁i', '▁a', 'or', 'I', '▁g', 'E', 's', 'm', 'A', '▁f', 'L', 't', 'F', 'ng', 'er', 'H', '▁v', 'J', 'z', '▁för', 'u', '▁l', 'ar', 'V', ':', 'ck', 'ade', 'K', '▁s', '–', '▁var', 'j', 'S', 'en', 'B', '8', '▁u', 'N', 'P'}
[('▁', 's'), ('d', 'e'), ('▁', 'h'), ('e', 'n'), ('a', 'n'), ('t', 't'), ('a', 'r'), ('▁', 'v'), ('▁', 'f'), ('▁', 'a'), ('o', 'm'), ('o', 'n'), ('l', 'l'), ('▁', 'de'), ('▁', 'm'), ('ö', 'r'), ('▁', 'o'), ('c', 'h'), ('▁', 'b'), ('a', 'de'), ('▁', 'k'), ('▁', 't'), ('i', 'g'), ('▁a', 'tt'), ('e', 'r'), ('n', 'g'), ('▁o', 'ch'), ('s', 't'),

6) Tokenize the corpus using the final vocabulary

In [10]:
def tokenize(corpus: list[str], merge_ops: list[tuple[str, str]]) -> list[str]:
    for pair in merge_ops:
        corpus = merge_bigrams(corpus, pair)
    return corpus

#### **TESTING**

Now we will compare the results of our implementation with the `sentencepiece` library. They should be the same.

In [11]:
assert tokenize(corpus_l, merge_ops) == sp.encode(corpus_raw, out_type=str)

## Building a Unigram tokenizer

**Author:** Diego Besada

Created using [Pierre Nugues' template](https://github.com/pnugues/edan20/blob/master/labs_2026/3-BPE.ipynb)

In [12]:
import math
from collections import Counter
from functools import cache

#### Unigram (from `sentencepiece` library)

As with BPE, we train a reference unigram model with `sentencepiece` for comparison.

In [13]:
_ = spm.SentencePieceTrainer.train(
    input='Selma/herrgard.txt', 
    model_prefix='m', 
    vocab_size=116,
    model_type='UNIGRAM', 
    user_defined_symbols='0,1,2,3,4,5,6,7,8,9',
    minloglevel=3
)

sp = spm.SentencePieceProcessor()
_ = sp.load('m.model')

sp.encode('Selma Lagerlöf')
sp.encode('Selma Lagerlöf', out_type=str)

[13, 110, 21, 23, 32, 14, 13, 96, 14, 22, 44, 23, 37, 48]

['▁', 'S', 'e', 'l', 'm', 'a', '▁', 'L', 'a', 'g', 'er', 'l', 'ö', 'f']

#### Unigram design

Instead of merging pairs, the unigram model scores every possible segmentation by the sum of its token log-probabilities and keeps the most likely one.

`unigram_lm` estimates a log-probability for each token from its frequency in the tokenized corpus; tokens that never occur (members of the seed vocabulary) are assigned a large negative log-probability.

In [14]:
def unigram_lm(tokenized_corpus):
    counts = Counter(tokenized_corpus)
    total = len(tokenized_corpus)
    unigram_probs = {token: math.log(count / total) for token, count in counts.items()}

    for token in vocabulary:
        if token not in unigram_probs:
            unigram_probs[token] = -1000

    return unigram_probs

tokenized_corpus = tokenize(corpus_l, merge_ops)
unigram_probs = unigram_lm(tokenized_corpus)

`tokenize_lm` finds the highest log-probability segmentation of a space-free character sequence with a cached recursive search over all the possible splits.

In [15]:
def tokenize_lm(char_seq: str, unigram_probs: dict[str, float]) -> tuple[float, list[str]]:
    '''Returns the highest log-probability segmentation of char_seq as (score, tokens).'''

    @cache
    def best_segmentation(suffix):
        if not suffix:
            return 0.0, []

        candidates = []
        for i in range(1, len(suffix) + 1):
            first, rest = suffix[:i], suffix[i:]
            first_prob = unigram_probs.get(first, -1000)
            rest_prob, rest_tokens = best_segmentation(rest)
            candidates.append((first_prob + rest_prob, [first] + rest_tokens))

        return max(candidates)

    return best_segmentation(char_seq)

tokenize_lm('▁senare', unigram_probs)
tokenize_lm('▁H', unigram_probs)

(-15.350338240623348, ['▁s', 'en', 'ar', 'e'])

(-5.056322032284403, ['▁H'])

`tokenize_text_lm` applies the same segmentation word by word, using the `▁` marker to split the corpus into words, and returns the total corpus log-likelihood together with the tokens.

In [16]:
def tokenize_text_lm(corpus: str, unigram_probs: dict[str, float]) -> tuple[float, list[str]]:
    re_token = '▁[^▁]+'
    tokenized_corpus = []
    corpus_prob = 0.0
    for word in re.finditer(re_token, corpus):
        prob, tokens = tokenize_lm(word.group(), unigram_probs)
        tokenized_corpus += tokens
        corpus_prob += prob
    return corpus_prob, tokenized_corpus

corpus_prob, tokenized_corpus = tokenize_text_lm(''.join(corpus), unigram_probs)
tokenized_corpus[:15]
corpus_prob

['▁', 'S', 'e', 'l', 'm', 'a', '▁', 'L', 'a', 'g', 'er', 'l', 'ö', 'f', '▁']

-494901.40131910506

#### Vocabulary selection

We re-estimate the token probabilities and keep the subwords whose removal would cost the least in corpus log-likelihood (an expectation–maximization loop).

In [17]:
unigram_logprobs_lm = unigram_probs.copy()
for _ in range(5):
    init_loglikelihood, tokens = tokenize_text_lm(corpus, unigram_logprobs_lm)
    unigram_logprobs_lm = unigram_lm(tokens)
    init_loglikelihood

-494901.40131910506

-494766.45666317357

-494584.21271454333

-494583.24282243924

-494583.24282243924

We now remove one subword at a time and measure how much the log-likelihood of the corpus drops without it. The subword that costs the least is the best candidate to be discarded.


In [18]:
logloss_word = []
for word in sorted(vocabulary, key=lambda w: -unigram_logprobs_lm[w]):
    if len(word) == 1:
        continue
    prob_word = unigram_logprobs_lm.pop(word)
    loglikelihood, _ = tokenize_text_lm(corpus, unigram_logprobs_lm)
    log_loss = init_loglikelihood - loglikelihood
    logloss_word.append((log_loss, word))
    unigram_logprobs_lm[word] = prob_word 

In [19]:
sorted(logloss_word)[:10]
out_candidate = min(logloss_word)[1]
out_candidate

[(78.0181036858121, '▁o'),
 (577.6622520053061, '▁a'),
 (797.1530523666297, '▁de'),
 (915.3494101668475, '▁en'),
 (1005.5210829891148, '▁u'),
 (1095.4463402613183, 'na'),
 (1284.3070624113898, '▁n'),
 (1331.218635343248, 'tt'),
 (1359.931525929307, 'ch'),
 (1556.2041463352507, '▁e')]

'▁o'

#### The tokenization without the subword


In [20]:
vocabulary.discard(out_candidate)
_ = unigram_logprobs_lm.pop(out_candidate)
for _ in range(5):
    new_loglikelihood, tokens = tokenize_text_lm(corpus, unigram_logprobs_lm)
    unigram_logprobs_lm = unigram_lm(tokens)
    new_loglikelihood, init_loglikelihood,init_loglikelihood - new_loglikelihood 

(-494661.26092612505, -494583.24282243924, 78.0181036858121)

(-494534.0921189297, -494583.24282243924, -49.15070350951282)

(-494534.0921189297, -494583.24282243924, -49.15070350951282)

(-494534.0921189297, -494583.24282243924, -49.15070350951282)

(-494534.0921189297, -494583.24282243924, -49.15070350951282)